In [22]:
import numpy as np
import pickle
from collections import Counter

np.random.seed(42)

print("Random Forest + Stacking")
print("="*60)

Random Forest + Stacking


## 1. Chargement et Préparation

In [23]:
print("\n[1/7] Chargement...")

with open('train_data.pkl', 'rb') as f:
    train_data = pickle.load(f)
with open('test_data.pkl', 'rb') as f:
    test_data = pickle.load(f)

X_train_raw = np.array(train_data['images'])
y_train_raw = np.array(train_data['labels']).flatten()
X_test_raw = np.array(test_data['images'])

print(f"Train: {X_train_raw.shape}")
print(f"Test: {X_test_raw.shape}")
print(f"Distribution: {dict(Counter(y_train_raw))}")

# Normalisation
X_train_flat = X_train_raw.reshape(len(X_train_raw), -1).astype(np.float32) / 255.0
X_test_flat = X_test_raw.reshape(len(X_test_raw), -1).astype(np.float32) / 255.0

# Split: 60% train, 20% validation, 20% pour meta-modèle
n = len(X_train_flat)
idx = np.random.permutation(n)

n_train = int(n * 0.6)
n_val = int(n * 0.2)

idx_train = idx[:n_train]
idx_val = idx[n_train:n_train+n_val]
idx_meta = idx[n_train+n_val:]

X_train = X_train_flat[idx_train]
y_train = y_train_raw[idx_train]

X_val = X_train_flat[idx_val]
y_val = y_train_raw[idx_val]

X_meta = X_train_flat[idx_meta]
y_meta = y_train_raw[idx_meta]

print(f"\nSplit: Train={len(X_train)}, Val={len(X_val)}, Meta={len(X_meta)}")


[1/7] Chargement...
Train: (1080, 28, 28, 3)
Test: (400, 28, 28, 3)
Distribution: {np.uint8(0): 486, np.uint8(4): 66, np.uint8(3): 194, np.uint8(2): 206, np.uint8(1): 128}

Split: Train=648, Val=216, Meta=216


## 2. PCA

In [24]:
print("\n[2/7] PCA...")

class PCA:
    def __init__(self, n_components):
        self.n_components = n_components
    
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        X_c = X - self.mean
        cov = np.cov(X_c, rowvar=False)
        vals, vecs = np.linalg.eigh(cov)
        idx = np.argsort(vals)[::-1]
        self.components = vecs[:, idx[:self.n_components]]
    
    def transform(self, X):
        return np.dot(X - self.mean, self.components)

pca = PCA(n_components=150)
pca.fit(X_train)

X_train_pca = pca.transform(X_train)
X_val_pca = pca.transform(X_val)
X_meta_pca = pca.transform(X_meta)
X_test_pca = pca.transform(X_test_flat)

print(f"  PCA: {X_train_pca.shape[1]} composantes")


[2/7] PCA...
  PCA: 150 composantes


## 3. Random Forest (Implémentation NumPy)

In [25]:
print("\n[3/7] Random Forest...")

class DecisionTree:
    def __init__(self, max_depth=10, min_samples=5):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.tree = None
    
    def gini(self, y):
        """Calcul Gini impurity"""
        if len(y) == 0:
            return 0
        probs = np.bincount(y) / len(y)
        return 1 - np.sum(probs ** 2)
    
    def best_split(self, X, y):
        """Trouve meilleure séparation"""
        best_gain = 0
        best_feature = None
        best_threshold = None
        
        current_gini = self.gini(y)
        n_features = X.shape[1]
        
        # Essayer features aléatoires (Random Forest)
        features = np.random.choice(n_features, size=min(20, n_features), replace=False)
        
        for feature in features:
            thresholds = np.percentile(X[:, feature], [25, 50, 75])
            
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) < self.min_samples or np.sum(right_mask) < self.min_samples:
                    continue
                
                left_gini = self.gini(y[left_mask])
                right_gini = self.gini(y[right_mask])
                
                n_left = np.sum(left_mask)
                n_right = np.sum(right_mask)
                weighted_gini = (n_left * left_gini + n_right * right_gini) / len(y)
                
                gain = current_gini - weighted_gini
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_gain
    
    def build_tree(self, X, y, depth=0):
        """Construction récursive de l'arbre"""
        # Conditions d'arrêt
        if depth >= self.max_depth or len(y) < self.min_samples or len(np.unique(y)) == 1:
            return {'leaf': True, 'value': np.bincount(y).argmax()}
        
        # Trouver meilleure séparation
        feature, threshold, gain = self.best_split(X, y)
        
        if feature is None or gain == 0:
            return {'leaf': True, 'value': np.bincount(y).argmax()}
        
        # Séparer
        left_mask = X[:, feature] <= threshold
        right_mask = ~left_mask
        
        # Récursion
        left_tree = self.build_tree(X[left_mask], y[left_mask], depth + 1)
        right_tree = self.build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return {
            'leaf': False,
            'feature': feature,
            'threshold': threshold,
            'left': left_tree,
            'right': right_tree
        }
    
    def fit(self, X, y):
        self.tree = self.build_tree(X, y)
    
    def predict_one(self, x, tree):
        """Prédiction pour un exemple"""
        if tree['leaf']:
            return tree['value']
        
        if x[tree['feature']] <= tree['threshold']:
            return self.predict_one(x, tree['left'])
        else:
            return self.predict_one(x, tree['right'])
    
    def predict(self, X):
        return np.array([self.predict_one(x, self.tree) for x in X])

class RandomForest:
    def __init__(self, n_trees=100, max_depth=50):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []
    
    def fit(self, X, y):
        print(f"  Entraînement {self.n_trees} arbres...")
        for i in range(self.n_trees):
            if i % 2 == 0:
                print(f"    Arbre {i+1}/{self.n_trees}")
            
            # Bootstrap
            idx = np.random.choice(len(X), size=len(X), replace=True)
            X_boot = X[idx]
            y_boot = y[idx]
            
            # Entraîner arbre
            tree = DecisionTree(max_depth=self.max_depth)
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)
    
    def predict(self, X):
        # Prédictions de tous les arbres
        predictions = np.array([tree.predict(X) for tree in self.trees])
        # Vote majoritaire
        return np.array([np.bincount(predictions[:, i]).argmax() for i in range(X.shape[0])])

# Entraînement Random Forest
rf = RandomForest(n_trees=15, max_depth=8)
rf.fit(X_train_pca, y_train)

# Validation
preds_rf_val = rf.predict(X_val_pca)
acc_rf = np.mean(preds_rf_val == y_val) * 100
print(f"\n  ✓ Random Forest - Validation: {acc_rf:.2f}%")

# Prédictions pour stacking
preds_rf_meta = rf.predict(X_meta_pca)
preds_rf_test = rf.predict(X_test_pca)


[3/7] Random Forest...
  Entraînement 15 arbres...
    Arbre 1/15
    Arbre 3/15
    Arbre 5/15
    Arbre 7/15
    Arbre 9/15
    Arbre 11/15
    Arbre 13/15
    Arbre 15/15

  ✓ Random Forest - Validation: 46.76%


## 4. Modèles de Base Supplémentaires

In [26]:
print("\n[4/7] Autres modèles de base...")

def relu(x):
    return np.maximum(0, x)

def softmax(x):
    x_max = np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def to_onehot(y, nc=5):
    oh = np.zeros((len(y), nc))
    oh[np.arange(len(y)), y] = 1
    return oh

# === Modèle 1: Logistic Regression ===
print("  Logistic Regression...")
D, C = X_train_pca.shape[1], 5
W_lr = np.random.randn(D, C) * 0.01
b_lr = np.zeros(C)
y_train_oh = to_onehot(y_train)

for epoch in range(1000):
    logits = np.dot(X_train_pca, W_lr) + b_lr
    probs = softmax(logits)
    loss = -np.mean(np.sum(y_train_oh * np.log(probs + 1e-8), axis=1))
    dlogits = (probs - y_train_oh) / len(X_train_pca)
    W_lr -= 0.1 * (np.dot(X_train_pca.T, dlogits) + 0.01 * W_lr)
    b_lr -= 0.1 * np.sum(dlogits, axis=0)

preds_lr_val = np.argmax(softmax(np.dot(X_val_pca, W_lr) + b_lr), axis=1)
preds_lr_meta = np.argmax(softmax(np.dot(X_meta_pca, W_lr) + b_lr), axis=1)
preds_lr_test = np.argmax(softmax(np.dot(X_test_pca, W_lr) + b_lr), axis=1)
acc_lr = np.mean(preds_lr_val == y_val) * 100
print(f"    Validation: {acc_lr:.2f}%")

# === Modèle 2: Neural Network 2 couches ===
print("  Neural Network...")
H = 256
W1_nn = np.random.randn(D, H) * np.sqrt(2.0 / D)
b1_nn = np.zeros(H)
W2_nn = np.random.randn(H, C) * np.sqrt(2.0 / H)
b2_nn = np.zeros(C)

for epoch in range(300):
    h = relu(np.dot(X_train_pca, W1_nn) + b1_nn)
    logits = np.dot(h, W2_nn) + b2_nn
    probs = softmax(logits)
    loss = -np.mean(np.sum(y_train_oh * np.log(probs + 1e-8), axis=1))
    
    dlogits = (probs - y_train_oh) / len(X_train_pca)
    dW2 = np.dot(h.T, dlogits) + 0.001 * W2_nn
    db2 = np.sum(dlogits, axis=0)
    dh = np.dot(dlogits, W2_nn.T)
    dh[h <= 0] = 0
    dW1 = np.dot(X_train_pca.T, dh) + 0.001 * W1_nn
    db1 = np.sum(dh, axis=0)
    
    W2_nn -= 0.01 * dW2
    b2_nn -= 0.01 * db2
    W1_nn -= 0.01 * dW1
    b1_nn -= 0.01 * db1

h_val = relu(np.dot(X_val_pca, W1_nn) + b1_nn)
preds_nn_val = np.argmax(softmax(np.dot(h_val, W2_nn) + b2_nn), axis=1)

h_meta = relu(np.dot(X_meta_pca, W1_nn) + b1_nn)
preds_nn_meta = np.argmax(softmax(np.dot(h_meta, W2_nn) + b2_nn), axis=1)

h_test = relu(np.dot(X_test_pca, W1_nn) + b1_nn)
preds_nn_test = np.argmax(softmax(np.dot(h_test, W2_nn) + b2_nn), axis=1)

acc_nn = np.mean(preds_nn_val == y_val) * 100
print(f"    Validation: {acc_nn:.2f}%")

# === Modèle 3: KNN ===
print("  KNN (k=5)...")
def knn_predict(X_train, y_train, X_test, k=5):
    preds = []
    for x in X_test:
        dists = np.sum((X_train - x)**2, axis=1)
        nearest = np.argsort(dists)[:k]
        votes = y_train[nearest]
        preds.append(np.bincount(votes).argmax())
    return np.array(preds)

preds_knn_val = knn_predict(X_train_pca, y_train, X_val_pca, k=5)
preds_knn_meta = knn_predict(X_train_pca, y_train, X_meta_pca, k=5)
preds_knn_test = knn_predict(X_train_pca, y_train, X_test_pca, k=5)

acc_knn = np.mean(preds_knn_val == y_val) * 100
print(f"    Validation: {acc_knn:.2f}%")

print("\n  ✓ 4 modèles de base entraînés")


[4/7] Autres modèles de base...
  Logistic Regression...
    Validation: 50.00%
  Neural Network...
    Validation: 48.61%
  KNN (k=5)...
    Validation: 43.06%

  ✓ 4 modèles de base entraînés


## 5. Stacking - Meta-Modèle

In [27]:
print("\n[5/7] Stacking - Meta-modèle...")

# Créer features pour meta-modèle (prédictions des modèles de base)
X_meta_features = np.column_stack([
    preds_rf_meta,
    preds_lr_meta,
    preds_nn_meta,
    preds_knn_meta
])

X_val_features = np.column_stack([
    preds_rf_val,
    preds_lr_val,
    preds_nn_val,
    preds_knn_val
])

X_test_features = np.column_stack([
    preds_rf_test,
    preds_lr_test,
    preds_nn_test,
    preds_knn_test
])

print(f"  Features meta: {X_meta_features.shape}")

# Entraîner meta-modèle (simple logistic regression)
print("  Entraînement meta-modèle...")
D_meta = X_meta_features.shape[1]
W_meta = np.random.randn(D_meta, C) * 0.01
b_meta = np.zeros(C)
y_meta_oh = to_onehot(y_meta)

for epoch in range(500):
    logits = np.dot(X_meta_features, W_meta) + b_meta
    probs = softmax(logits)
    loss = -np.mean(np.sum(y_meta_oh * np.log(probs + 1e-8), axis=1))
    
    dlogits = (probs - y_meta_oh) / len(X_meta_features)
    W_meta -= 0.1 * np.dot(X_meta_features.T, dlogits)
    b_meta -= 0.1 * np.sum(dlogits, axis=0)

# Prédictions avec stacking
preds_stack_val = np.argmax(softmax(np.dot(X_val_features, W_meta) + b_meta), axis=1)
preds_stack_test = np.argmax(softmax(np.dot(X_test_features, W_meta) + b_meta), axis=1)

acc_stack = np.mean(preds_stack_val == y_val) * 100
print(f"\n  ✓ Stacking - Validation: {acc_stack:.2f}%")


[5/7] Stacking - Meta-modèle...
  Features meta: (216, 4)
  Entraînement meta-modèle...

  ✓ Stacking - Validation: 49.07%


## 6. Vote Majoritaire Simple

In [19]:
print("\n[6/7] Vote majoritaire...")

# Vote simple
all_preds_val = np.vstack([preds_rf_val, preds_lr_val, preds_nn_val, preds_knn_val])
preds_vote_val = np.array([np.bincount(all_preds_val[:, i]).argmax() for i in range(len(y_val))])

all_preds_test = np.vstack([preds_rf_test, preds_lr_test, preds_nn_test, preds_knn_test])
preds_vote_test = np.array([np.bincount(all_preds_test[:, i]).argmax() for i in range(len(X_test_pca))])

acc_vote = np.mean(preds_vote_val == y_val) * 100
print(f"  ✓ Vote - Validation: {acc_vote:.2f}%")


[6/7] Vote majoritaire...
  ✓ Vote - Validation: 49.07%


## 7. Comparaison et Sélection du Meilleur

In [28]:
print("\n" + "="*60)
print("COMPARAISON DES MODÈLES")
print("="*60)

models = [
    ("Random Forest", acc_rf, preds_rf_test),
    ("Logistic Regression", acc_lr, preds_lr_test),
    ("Neural Network", acc_nn, preds_nn_test),
    ("KNN", acc_knn, preds_knn_test),
    ("Stacking (Meta)", acc_stack, preds_stack_test),
    ("Vote Majoritaire", acc_vote, preds_vote_test)
]

for name, acc, preds in models:
    dist = dict(Counter(preds))
    print(f"\n{name}")
    print(f"  Validation: {acc:.2f}%")
    print(f"  Distribution: {dist}")

# Sélectionner le meilleur
best_idx = np.argmax([acc for _, acc, _ in models])
best_name, best_acc, best_preds = models[best_idx]

print("\n" + "="*60)
print(f"MEILLEUR MODÈLE: {best_name}")
print(f"Précision validation: {best_acc:.2f}%")
print("="*60)


COMPARAISON DES MODÈLES

Random Forest
  Validation: 46.76%
  Distribution: {np.int64(0): 352, np.int64(3): 23, np.int64(1): 7, np.int64(2): 18}

Logistic Regression
  Validation: 50.00%
  Distribution: {np.int64(0): 265, np.int64(3): 69, np.int64(2): 57, np.int64(4): 1, np.int64(1): 8}

Neural Network
  Validation: 48.61%
  Distribution: {np.int64(0): 286, np.int64(3): 64, np.int64(2): 43, np.int64(1): 7}

KNN
  Validation: 43.06%
  Distribution: {np.int64(0): 247, np.int64(1): 37, np.int64(3): 53, np.int64(4): 8, np.int64(2): 55}

Stacking (Meta)
  Validation: 49.07%
  Distribution: {np.int64(0): 269, np.int64(2): 71, np.int64(1): 36, np.int64(3): 24}

Vote Majoritaire
  Validation: 49.07%
  Distribution: {np.int64(0): 324, np.int64(3): 42, np.int64(2): 32, np.int64(1): 2}

MEILLEUR MODÈLE: Logistic Regression
Précision validation: 50.00%


## 8. Génération Fichier Soumission

In [29]:
print("\n[7/7] Génération fichiers...")

# CSV
with open('submission.csv', 'w') as f:
    f.write('ID,Label\n')
    for i, pred in enumerate(best_preds, start=1):
        f.write(f'{i},{int(pred)}\n')

print("✓ submission.csv créé")

# PKL
submission_dict = {'predictions': best_preds.tolist()}
with open('submission.pkl', 'wb') as f:
    pickle.dump(submission_dict, f)

print("✓ submission.pkl créé")

# Vérification
with open('submission.csv', 'r') as f:
    lines = f.readlines()
    print(f"\nVérification:")
    print(f"  Lignes: {len(lines)}")
    print(f"  Premier: {lines[1].strip()}")
    print(f"  Dernier: {lines[-1].strip()}")

print("\n" + "="*60)
print("TERMINÉ! Random Forest + Stacking")
print("="*60)
print(f"\nMeilleur modèle: {best_name}")
print(f"Précision: {best_acc:.2f}%")
print(f"Distribution: {dict(Counter(best_preds))}")
print("\nFichiers prêts pour Kaggle!")


[7/7] Génération fichiers...
✓ submission.csv créé
✓ submission.pkl créé

Vérification:
  Lignes: 401
  Premier: 1,0
  Dernier: 400,2

TERMINÉ! Random Forest + Stacking

Meilleur modèle: Logistic Regression
Précision: 50.00%
Distribution: {np.int64(0): 265, np.int64(3): 69, np.int64(2): 57, np.int64(4): 1, np.int64(1): 8}

Fichiers prêts pour Kaggle!
